# Proyecto Parcial I: Predicción del rendimiento académico con aprendizaje automático
**Universidad de Guayaquil — Carrera de Ciencia de Datos e Inteligencia Artificial**

**Paralelo:** 4-2 | **Grupo:** E | **Fecha:** 25 de noviembre de 2026

**Integrantes:**
- Granda Zambrano Cecilia Lorena
- Almeida Terry
- Campoverde María Fernanda
- Molina Sarco Juan
- Núñez Katherine

**Dataset:** Student Performance, Matemáticas (`student-mat.csv`), Cortez y Silva (2008).

**Preguntas:** (1) ¿Qué nota final obtendrá un estudiante? (2) ¿Aprobará o reprobará?

---
**Archivo independiente:** Bono Ridge Lasso

In [1]:
# Librerías
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, precision_score,
                             recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
import statsmodels.api as sm

SEED = 2026  # semilla del grupo (justificación en la Parte 2.5)

In [2]:
# Carga del dataset: funciona en Google Colab, en el PC, en GitHub Codespaces y en GitHub Actions
ARCHIVO = 'student-mat.csv'
rutas = [ARCHIVO, os.path.join('data', ARCHIVO)]
ruta = next((r for r in rutas if os.path.exists(r)), None)

if ruta is None:
    try:                      # solo existe en Google Colab
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'Copie {ARCHIVO} en la misma carpeta del notebook.')
    print(f'Seleccione el archivo {ARCHIVO}')
    subido = files.upload()
    ruta = next(iter(subido))

df = pd.read_csv(ruta, sep=None, engine='python')  # detecta separador ',' o ';'
assert df.shape[1] == 33, 'El archivo no es el student-mat.csv esperado (33 columnas).'
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


### Preparación (necesaria para ejecutar esta parte sola)
El detalle y las justificaciones están en la Parte 2.

In [3]:
# Tipos de variables
num_cols = df.select_dtypes('number').columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

In [4]:
# 2.1 Faltantes y duplicados en el dataset original
print('Faltantes originales:', df.isnull().sum().sum())
print('Filas duplicadas    :', df.duplicated().sum())

# Introducir 5 % de faltantes al azar (sobre una copia, para no alterar el EDA)
datos = df.copy()
rng = np.random.default_rng(SEED)
n_nan = int(round(0.05 * len(datos)))
for col in ['absences', 'studytime']:
    datos[col] = datos[col].astype(float)
    datos.loc[rng.choice(datos.index, size=n_nan, replace=False), col] = np.nan
print(datos[['absences', 'studytime']].isnull().sum())
print((datos[['absences', 'studytime']].isnull().mean() * 100).round(2))
print(df[['absences', 'studytime']].agg(['mean', 'median', 'skew']).round(2))

Faltantes originales: 0
Filas duplicadas    : 0
absences     20
studytime    20
dtype: int64
absences     5.06
studytime    5.06
dtype: float64
        absences  studytime
mean        5.71       2.04
median      4.00       2.00
skew        3.67       0.63


In [5]:
# Variable objetivo de clasificación (Parte 4.1)
datos['aprueba'] = (datos['G3'] >= 10).astype(int)

# 2.5 División 80/20. Se usa la MISMA partición para todos los modelos.
train, test = train_test_split(datos, test_size=0.20, random_state=SEED)
print(f'Entrenamiento: {len(train)} | Prueba: {len(test)}')

Entrenamiento: 316 | Prueba: 79


In [6]:
# 2.3 y 2.4 Preprocesamiento dentro de Pipeline + ColumnTransformer
feat_A = [c for c in df.columns if c != 'G3']                 # incluye G1 y G2
feat_B = [c for c in feat_A if c not in ['G1', 'G2']]         # sin G1 ni G2

def crear_pipeline(features, modelo):
    """Imputa, escala y codifica. Todo se ajusta solo con los datos de entrenamiento."""
    nums = [c for c in features if c in num_cols]
    cats = [c for c in features if c in cat_cols]
    prep = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler', StandardScaler())]), nums),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          # drop='first' evita la multicolinealidad perfecta entre dummies
                          ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))]), cats)
    ], sparse_threshold=0)
    return Pipeline([('preprocessor', prep), ('model', modelo)])

y_train, y_test = train['G3'], test['G3']

In [7]:
# Entrenar el Modelo B (se explica en la Parte 3)
modelos, preds = {}, {}
modelos['B'] = crear_pipeline(feat_B, LinearRegression()).fit(train[feat_B], y_train)
preds['B'] = modelos['B'].predict(test[feat_B])
nombres = [n.split('__')[1] for n in modelos['B'].named_steps['preprocessor'].get_feature_names_out()]

## Bonificación. Ridge y Lasso frente al Modelo B

In [8]:
# Bono: Ridge y Lasso. Alfa se elige por validación cruzada SOLO en entrenamiento.
from sklearn.model_selection import GridSearchCV, KFold
cv = KFold(5, shuffle=True, random_state=SEED)
filas = [{'Modelo': 'Lineal B', 'alpha': np.nan, 'MSE': mean_squared_error(y_test, preds['B']),
          'R2': r2_score(y_test, preds['B'])}]
ajustados = {}
for nombre, est, alfas in [('Ridge', Ridge(), np.logspace(-2, 3, 60)),
                           ('Lasso', Lasso(max_iter=20000), np.logspace(-3, 0, 60))]:
    g = GridSearchCV(crear_pipeline(feat_B, est), {'model__alpha': alfas}, cv=cv,
                     scoring='neg_mean_squared_error').fit(train[feat_B], y_train)
    p = g.predict(test[feat_B]); ajustados[nombre] = g.best_estimator_
    filas.append({'Modelo': nombre, 'alpha': g.best_params_['model__alpha'],
                  'MSE': mean_squared_error(y_test, p), 'R2': r2_score(y_test, p)})
tabla_bono = pd.DataFrame(filas)
tabla_bono['RMSE'] = np.sqrt(tabla_bono['MSE'])
display(tabla_bono.round(4))

coef_lasso = pd.Series(ajustados['Lasso'].named_steps['model'].coef_, index=nombres)
eliminadas = coef_lasso[coef_lasso == 0].index.tolist()
print(f'Lasso elimina {len(eliminadas)} de {len(nombres)} columnas:\n{eliminadas}')
print(f'\nConserva:\n{coef_lasso[coef_lasso != 0].sort_values().round(3).to_string()}')

,Modelo,alpha,MSE,R2,RMSE
0,Lineal B,NaN,18.1722,0.0357,4.2629
1,Ridge,96.1725,16.5147,0.1236,4.0638
2,Lasso,0.1727,16.8636,0.1051,4.1065


Lasso elimina 25 de 39 columnas:
['Fedu', 'studytime', 'Dalc', 'Walc', 'school_MS', 'address_U', 'Pstatus_T', 'Mjob_health', 'Mjob_other', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other', 'schoolsup_yes', 'paid_yes', 'activities_yes', 'nursery_yes', 'higher_yes', 'internet_yes']

Conserva:
failures        -1.381
age             -0.303
romantic_yes    -0.301
goout           -0.236
health          -0.120
famsup_yes      -0.030
traveltime      -0.027
freetime         0.110
absences         0.110
famrel           0.155
famsize_LE3      0.269
Mjob_services    0.403
Medu             0.415
sex_M            0.534


**¿Mejoran el R²?** Sí, en esta partición: pasa de 0.036 (lineal) a **0.124 con Ridge** y **0.105 con Lasso**. La regularización reduce el sobreajuste, aunque el desempeño sigue siendo bajo. Con una sola partición de 79 casos no se puede asegurar que la mejora se mantenga con otros datos.

**¿Qué elimina Lasso?** 25 de las 39 columnas codificadas; la salida anterior las lista. Entre ellas están `studytime`, `Fedu`, `Dalc`, `Walc`, `higher_yes`, `schoolsup_yes`, `internet_yes` y varias dummies de ocupación, motivo y tutor. Un coeficiente 0 no prueba que la variable sea irrelevante: depende de alfa, de la muestra y de la correlación con otras variables. En las categóricas se eliminan dummies, no necesariamente la variable original completa.

## Referencias
Cortez, P., & Silva, A. (2008). Using data mining to predict secondary school student performance. En A. Brito & J. Teixeira (Eds.), *Proceedings of 5th Future Business Technology Conference (FUBUTEC 2008)* (pp. 5–12). EUROSIS.

> **Pendiente del grupo:** agregar en APA 7 las fuentes que realmente hayan consultado en la biblioteca de la Universidad.

## Declaración de uso de inteligencia artificial
Se utilizó Claude (Anthropic) para revisar el notebook frente a la guía, corregir y completar el código, ejecutarlo y redactar un borrador de las interpretaciones y conclusiones. El grupo debe revisar el contenido, redactar las respuestas con sus propias palabras y poder explicar cada línea en la defensa.